In [1]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

In [3]:
X = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\X_train_road_3.csv')
S = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\X_train_speed_3.csv')
y = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\y_train_3.csv')

In [4]:
X_train = torch.tensor(X.values, dtype=torch.float32).to(device='cuda')
S_train = torch.tensor(S.values, dtype=torch.float32).to(device='cuda')
y_tensor = torch.tensor(y.values, dtype=torch.float32).to(device='cuda')

In [5]:
print(len(X_train))
print(len(S_train))
print(len(y_tensor))

30650
30650
30650


In [6]:
X_tensor = torch.cat((X_train, S_train), dim=1)
print(X_tensor[0])
print(X_tensor[0].size())

tensor([0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.4882], device='cuda:0')
torch.Size([12289])


In [7]:
print(y_tensor[0])
print(y_tensor[0].size())

tensor([-0.0044, -0.0028], device='cuda:0')
torch.Size([2])


In [8]:
dataset = TensorDataset(X_tensor, y_tensor)
data_loader = DataLoader(dataset, batch_size=512, shuffle=False)

In [9]:
class FeedforwardNet(nn.Module):
    def __init__(self, input_size=12289, hidden_size_1=1024, hidden_size_2=512, hidden_size_3=512,
                 output_size=2):
        super(FeedforwardNet, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size_1)
        self.fc2 = nn.Linear(hidden_size_1, hidden_size_2)
        self.fc3 = nn.Linear(hidden_size_2, hidden_size_3)
        self.fc4 = nn.Linear(hidden_size_3, output_size)
        self.relu = nn.ReLU()
        self.tanh = nn.Tanh()

    def forward(self, input_data):
        out = self.fc1(input_data)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.relu(out)
        out = self.fc3(out)
        out = self.relu(out)
        out = self.fc4(out)
        out = self.tanh(out)
        return out

In [10]:
model = FeedforwardNet().cuda()

In [11]:
# Функция для расчета L1 штрафа
def l1_penalty(params):
    return sum(p.abs().sum() for p in params)

# Функция для расчета L2 штрафа
def l2_penalty(params):
    return sum(p.pow(2.0).sum() for p in params)

# Обучение модели с L1 и L2 регуляризацией
criterion = nn.L1Loss()
optimizer = optim.Adam(model.parameters(), lr=0.0000001, weight_decay=0.0001)
l1_factor = 1e-5
l2_factor = 1e-5


In [16]:
# Обучение модели
num_epochs = 50
for epoch in range(num_epochs):
    for batch_x, batch_y in data_loader:  # Итерация по батчам данных
        optimizer.zero_grad()  # Обнуление градиентов

        outputs = model(batch_x.unsqueeze(0))  # Передача входных данных через модель
        loss = criterion(outputs, batch_y)  # Вычисление потерь
        
        l1_loss = l1_penalty(model.parameters())
        l2_loss = l2_penalty(model.parameters())
        total_loss = loss + l1_factor * l1_loss + l2_factor * l2_loss
        
        loss.backward()  # Обратное распространение ошибки
        optimizer.step()  # Обновление весов модели

    print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.9f}')

C:\Users\filip\anaconda3\envs\ETS_autopylot\Lib\site-packages\torch\nn\modules\loss.py:101: UserWarning: Using a target size (torch.Size([512, 2])) that is different to the input size (torch.Size([1, 512, 2])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.l1_loss(input, target, reduction=self.reduction)
C:\Users\filip\anaconda3\envs\ETS_autopylot\Lib\site-packages\torch\nn\modules\loss.py:101: UserWarning: Using a target size (torch.Size([442, 2])) that is different to the input size (torch.Size([1, 442, 2])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.l1_loss(input, target, reduction=self.reduction)


Epoch [1/50], Loss: 0.002689489
Epoch [2/50], Loss: 0.002676146
Epoch [3/50], Loss: 0.002661764
Epoch [4/50], Loss: 0.002648680
Epoch [5/50], Loss: 0.002635143
Epoch [6/50], Loss: 0.002622084
Epoch [7/50], Loss: 0.002610141
Epoch [8/50], Loss: 0.002595728
Epoch [9/50], Loss: 0.002583341
Epoch [10/50], Loss: 0.002572482
Epoch [11/50], Loss: 0.002561082
Epoch [12/50], Loss: 0.002548879
Epoch [13/50], Loss: 0.002536452
Epoch [14/50], Loss: 0.002525130
Epoch [15/50], Loss: 0.002513039
Epoch [16/50], Loss: 0.002500725
Epoch [17/50], Loss: 0.002490308
Epoch [18/50], Loss: 0.002478431
Epoch [19/50], Loss: 0.002467050
Epoch [20/50], Loss: 0.002455080
Epoch [21/50], Loss: 0.002445081
Epoch [22/50], Loss: 0.002433663
Epoch [23/50], Loss: 0.002423364
Epoch [24/50], Loss: 0.002411759
Epoch [25/50], Loss: 0.002401290
Epoch [26/50], Loss: 0.002389636
Epoch [27/50], Loss: 0.002378783
Epoch [28/50], Loss: 0.002368662
Epoch [29/50], Loss: 0.002358414
Epoch [30/50], Loss: 0.002348681
Epoch [31/50], Loss

In [17]:
torch.save(model.state_dict(), 'C:\PycharmProjects\ETS_Autopilot\static\weight_model\weight_wheel_nn_forward_6.pth')